In [0]:
booking = spark.read.table("airline_operations.airline.bookings_transformed")
flight = spark.read.table("airline_operations.airline.flights_transformed")
passenger_preference = spark.read.table("airline_operations.airline.passenger_preferences_clean")

# ... rest of your analytical report and display code block ...
from pyspark.sql.functions import sum, col, rank, concat, lit
from pyspark.sql.window import Window

# Airline Revenue Report
airline_revenue_df = booking.join(flight, booking.flight_id == flight.flight_id) \
    .groupBy("airline").agg(sum("revenue").alias("revenue"))
display(airline_revenue_df.select("airline", "revenue"))

# Route Performance Report
flight=flight.withColumn("route", concat(flight.from_city, lit(" to "), flight.to_city))
route_revenue_df = booking.join(flight, booking.flight_id == flight.flight_id) \
    .groupBy("route").agg(sum("revenue").alias("revenue"))
display(route_revenue_df.select("route", "revenue"))

# Passenger Preference Report
# Assuming meal and seat columns exist in booking table
passenger_pref_df = passenger_preference.select("passenger_name", "meal", "seat")
display(passenger_pref_df)

# Flight Delay Report
# Assuming status column exists in flight table
flight_status_df = flight.select("flight_id", "status")
display(flight_status_df)

# Top Revenue Flights
top_flights_df = booking.groupBy("flight_id").agg(sum("revenue").alias("revenue"))
window_spec = Window.orderBy(col("revenue").desc())
top_flights_df = top_flights_df.withColumn("rank", rank().over(window_spec))
display(top_flights_df.select("flight_id", "rank").orderBy("rank"))

airline,revenue
Air India,68000.0
Akasa,62000.0
Indigo,90000.0
Vistara,71500.0


route,revenue
Mumbai to Chennai,9000.0
Delhi to Mumbai,7500.0
Delhi to Hyderabad,18000.0
Chennai to Bangalore,25000.0
Bangalore to Mumbai,23500.0
Pune to Delhi,10000.0
Hyderabad to Delhi,39000.0
Goa to Delhi,7000.0
Chennai to Pune,24000.0
Bangalore to Hyderabad,38000.0


passenger_name,meal,seat
Rahul Sharma,Veg,Window
Priya Reddy,Non-Veg,Aisle
Amit Kumar,Veg,Middle
Sneha Patel,Jain,Window
Farhan Ali,Non-Veg,Aisle
Neha Singh,Veg,Window
Arjun Verma,Veg,Middle
Meera Nair,Jain,Window
Kiran Rao,Veg,Aisle
Nisha Reddy,Non-Veg,Window


flight_id,status
F101,On Time
F102,Delayed
F103,On Time
F104,Cancelled
F105,On Time
F106,Delayed
F107,On Time
F108,On Time
F109,Delayed
F110,On Time


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


flight_id,rank
F101,1
F103,2
F109,3
F107,4
F105,5
F113,6
F110,7
F115,8
F111,9
F114,10
